# Human-in-the-Loop: Forhåndsaksjonsporter, risikoklassifisering og revisjonslogg

README for denne leksjonen introduserer Human-in-the-Loop med et kort utdrag som ber brukeren om å `GODKJENNE` eller `AVSLÅ` etter at agenten allerede har produsert et svar. Det mønsteret er et fint utgangspunkt, men produksjons-HITL-implementeringer trenger vanligvis tre tilleggsbiter:

1. En **forhåndsaksjonsport** som kjører **før** agenten utfører et risikabelt steg, slik at kostnad, irreversibilitet og ventetid holdes under kontroll.
2. **Risikoklassifisering**, slik at handlinger med lav risiko auto-utføres, handlinger med middels risiko godkjennes i batch, og bare handlinger med høy risiko blokkeres av et menneske.
3. En **revisjonslogg pluss revisjonssløyfe**, slik at hver portbeslutning lagres som JSONL, og en avslag gir ny prompt til agenten med en strukturert grunn i stedet for bare å skrive ut `Revising...`.

Denne notebooken bygger hver av disse på toppen av de samme primitivene som `06-system-message-framework.ipynb`. Den kjører ende-til-ende i `DEMO_MODE = True` (ingen interaktiv input nødvendig) eller med reelle `input()`-forespørsler når `DEMO_MODE = False`. Merk: i DEMO_MODE er tredje mål sitt forsøk skriptet slik at sløyfemekanikkene er synlige ende-til-ende. Reell revisjonsdrevet re-klassifisering krever `DEMO_MODE = False` og en operatør.

**Utenfor omfang (håndteres i andre leksjoner):** autentisering og tilgangskontroll (Leksjon 06 README trussel 2), verktøys-kall mellomvare (Leksjon 14 MAF dypdykk), fler-agent debattmønstre.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Mønster 1: Forhåndsgodkjenningsport

README-eksempelet med HITL kaller først agenten, deretter ber brukeren om å godkjenne resultatet. Dette er en **etter-handling**-flyt. Agenten har allerede utført handlingen, så kostnaden for LLM-kallet er allerede betalt, og eventuelle bivirkninger (sendt e-post, skrevet databaseoppføring, lagt ut kommentar) har allerede skjedd.

En **forhånds-handling**-flyt setter porten inn før agenten utfører den risikable handlingen. Agenten foreslår handlingen, porten avgjør om den skal utføres, og bare ved godkjenning oppstår bivirkningen.

| Aspekt | Godkjenning etter handling (README-eksempel) | Forhåndsgodkjenningsport (denne notatboken) |
|---|---|---|
| Når skjer godkjenning? | Etter at agenten har produsert resultatet | Før noen bivirkning utføres |
| LLM-kostnad ved avslag | Allerede betalt | Betalt kun for forslaget, ikke handlingen |
| Uopprettelige bivirkninger | Mulig (handlingen har allerede skjedd) | Forhindret |
| Revisjonsklarhet | Godkjenning er en utskriftsmelding | Godkjenning er en JSON-post med tidsstempel, handling, årsak |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Mønster 2: Risikonivåer

Ikke alle handlinger trenger menneskelig godkjenning. Et kun-lese oppslag mot en offentlig API har andre innsatsnivåer enn å sende en kunde-e-post. Å behandle begge likt sløser med operatørens oppmerksomhet og senker agentens ytelse.

En enkel modell med 3 nivåer:

| Nivå | Eksempler | Godkjenningsflyt |
|---|---|---|
| `lav` (kun-lese) | Søk i en kunnskapsbase, se etter flyalternativer, hente en offentlig nettside | Auto-eksekvering, logget for revisjon |
| `medium` (billig endring) | Cache et resultat, øke en teller, planlegge en påminnelse | Auto-eksekvering, men samlet til daglig gjennomgang |
| `høy` (eksternrettet eller irreversibel) | Sende en e-post, belaste et kort, poste til en offentlig kanal | Blokkert til menneskelig godkjenning |

Dette er én måte å dele inn nivåer på. Produksjonssystemer bruker ofte mer detaljerte nivåer (f.eks. AWS IAM-tillatelser, rollebaserte tilgangsnivåer). Den 3-nivås versjonen nedenfor er den minste nyttige versjonen for en agent som blander kun-lese og bivirkende handlinger.

Klassifisereren nedenfor bruker nøkkelordheuristikker slik at demoen forblir deterministisk og billig. I et produksjonssystem ville du byttet denne ut med en lært klassifiserer eller en policy-motor.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Mønster 3: Revisjonslogg og revisjonssyklus

En `print("Response approved.")` er ikke en revisjonslogg. For tillit bør hver beslutning i porten registreres som en strukturert hendelse som du senere kan søke i, spille av på nytt, eller knytte til en hendelsesgjennomgang.

To deler:

1. **Append-only JSONL.** Én linje per beslutning, med tidsstempel, handling, nivå, beslutning, årsak. Lett å søke i med grep, lett å sende til en ekte logglagring senere.
2. **Revisjonssyklus ved avslag.** Når porten returnerer `deny`, spør agenten seg selv på nytt med avslagsårsaken i konteksten, slik at neste forslag kan unngå problemet.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Ytterligere ressurser

Flere andre offentlige prosjekter implementerer variasjoner av disse HITL-mønstrene. Sammenlign tilnærminger for å finne hva som passer til din stack:

- **LangChain** menneske-i-løkken verktøyinnpakninger ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): verktøyinnpakninger som stopper utførelsen for menneskelig inndata.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ omstrukturerte dette): bruker en agentrolle spesielt for å representere mennesket i multi-agent samtaler.
- **Semantic Kernel** funksjonsfiltre ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): mellomvare som kjører rundt hvert funksjonskall, egnet for styringslogikk.

Hvert prosjekt håndterer de tre delmønstrene forskjellig: LangChain pakker dem inn som verktøy, AutoGen bruker en agentrolle, Semantic Kernel bruker mellomvarefiltre. Les en eller to implementeringer i sin helhet før du velger et design for din egen agent.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
